# Мои шахматные партии

## Задача

Предсказание исхода партии по признакам:

- Оставшееся у меня время
- Разница оставшегося времени с соперником
- Номер хода
- Разница рейтингов

## Интерпретация

-> Сколько у меня должно оставаться времени на ходе `N`?  
В среднем, для идеально сбалансированной партии.

#### Как интерпретировать

Модель умеет предсказывать `P(win | sec, sec_diff, move_num, rate_diff)`  

- Фиксируем `sec_diff = 0`, `move_num = N`, `rate_diff = 0`
- Прогоняем через модель `sec` в диапазоне `[0 .. 300]`
- Получаем кривую `P(win)` в зависимости от оставшегося у меня времени `sec`
- Находим, где кривая пересекает порог `P(win) = 0.48` (консервативный ориентир)

## Подготовка окружения

```bash
./tools/install.sh
source .venv/bin/activate
python tools/get_data.py
```

In [8]:
import pandas as pd

import chess
import chess.pgn

## Подготовка данных

In [9]:
def get_game(path: str, index: int) -> str:
    with open(path) as f:
        for _ in range(index):
            chess.pgn.skip_game(f)
        return chess.pgn.read_game(f)

### Рассмотрим какую-нибудь игру

In [10]:
print(get_game("data.pgn", 0))

[Event "rated blitz game"]
[Site "https://lichess.org/Lnhx4IvX"]
[Date "2026.04.29"]
[Round "-"]
[White "qorovin"]
[Black "crazyhandrick"]
[Result "0-1"]
[GameId "Lnhx4IvX"]
[UTCDate "2026.04.29"]
[UTCTime "12:58:46"]
[WhiteElo "1309"]
[BlackElo "1297"]
[WhiteRatingDiff "-6"]
[BlackRatingDiff "+6"]
[Variant "Standard"]
[TimeControl "300+3"]
[ECO "C44"]
[Opening "Ponziani Opening"]
[Termination "Time forfeit"]

1. e4 { [%clk 0:05:00] } 1... e5 { [%clk 0:05:00] } 2. Nf3 { [%clk 0:05:02] } 2... Nc6 { [%clk 0:05:00] } 3. c3 { [%clk 0:05:05] } 3... d6 { [%clk 0:05:01] } 4. d4 { [%clk 0:05:00] } 4... Bg4 { [%clk 0:05:02] } 5. d5 { [%clk 0:04:59] } 5... Bxf3 { [%clk 0:05:04] } 6. Qxf3 { [%clk 0:04:57] } 6... Nce7 { [%clk 0:05:00] } 7. Bb5+ { [%clk 0:04:54] } 7... c6 { [%clk 0:05:01] } 8. dxc6 { [%clk 0:04:50] } 8... bxc6 { [%clk 0:05:02] } 9. Bc4 { [%clk 0:04:46] } 9... Nf6 { [%clk 0:05:00] } 10. Bg5 { [%clk 0:04:36] } 10... Ng6 { [%clk 0:04:58] } 11. Bxf6 { [%clk 0:04:30] } 11... gxf6 { [%cl

### Какие данные потребуются

| Признаки                | Изменения                                    |
|-------------------------|----------------------------------------------|
| `Variant`               |    Экземпляры только со значением `Standard` |
| `WhiteElo` + `BlackElo` | -> `rate_diff` (мой рейтинг минус соперника) |
| `White` + `Black`       | -> `color` (цвет моих фигур)                 |
| `Запись ходов`          | -> `moves [ (sec, sec_diff) , ... ]`         |
| `Result`                | -> `win / draw / loss` (one-hot)             |

### Получить `df`

In [11]:
import re

NICK    = "qorovin"
SEC_INI = 300

def parse_clock(comment):
    m = re.search(r'\[%clk 0:(\d+):(\d+)\]', comment)
    return int(m.group(1)) * 60 + int(m.group(2)) if m else None

rows = []

with open("data.pgn") as f:
    while True:

        game = chess.pgn.read_game(f)
        if game is None:
            break

        headers = game.headers

        if headers.get("Variant", "Standard") != "Standard":
            continue

        is_white  = (headers["White"] == NICK)
        my_color  = chess.WHITE if is_white else chess.BLACK
        opp_color = chess.BLACK if is_white else chess.WHITE

        result = headers["Result"]
        if result == "1/2-1/2":
            win, draw, loss = 0, 1, 0
        elif (result == "1-0") == is_white:
            win, draw, loss = 1, 0, 0
        else:
            win, draw, loss = 0, 0, 1

        rate_diff = (int(headers["WhiteElo"]) - int(headers["BlackElo"])) * (1 if is_white else -1)

        moves  = []
        clocks = {chess.WHITE: SEC_INI, chess.BLACK: SEC_INI}
        node   = game

        while node.variations:
            node  = node.variations[0]
            mover = node.parent.board().turn
            clk   = parse_clock(node.comment)

            if clk is not None:
                if mover == my_color:
                    moves.append((clk, clk - clocks[opp_color]))
                clocks[mover] = clk

        rows.append({
            "color":     int(is_white),
            "rate_diff": rate_diff,
            "win":       win,
            "draw":      draw,
            "loss":      loss,
            "moves":     moves,
        })

df = pd.DataFrame(rows)
df.shape

(2052, 6)

### `df`

In [12]:
pd.concat([df.head(2), df.tail(2)])

,color,rate_diff,win,draw,loss,moves
0,1,12,0,0,1,"[(300, 0), (302, 2), (305, 5), (300, -1), (299..."
1,0,-13,0,1,0,"[(300, 0), (301, 0), (301, 2), (304, 3), (297,..."
2050,1,-16,0,0,1,"[(300, 0), (296, -4), (288, -8), (267, 8), (23..."
2051,0,-21,0,0,1,"[(300, 0), (301, 1), (301, 0), (299, 16), (291..."
